# S&P 500 Statistical Diagnostics & Stylized Facts

This notebook extends the exploratory data analysis phase by formally testing the statistical properties of S&P 500 daily log returns.

The objective is to determine whether the return series satisfies the assumptions required for time series modeling and to identify key characteristics commonly observed in financial markets.

The analysis focuses on:

- Distribution properties
- Tail behavior
- Autocorrelation structure
- Volatility persistence
- Heteroskedasticity
- White noise behavior

These diagnostics provide the statistical foundation for later forecasting, volatility modeling, and anomaly detection phases.

In [26]:
# Import needed libraries
import pandas as pd 
import numpy as np 
# Setting Plotly backend plotting for pandas 
pd.options.plotting.backend = 'plotly'
import plotly.io as pio
pio.templates.default = 'plotly_dark'
import plotly.graph_objects as go
import plotly.express as px

import scipy.stats as stats
from scipy.stats import norm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

## Import Required Libraries

The following libraries are used for:

- Statistical testing
- Distribution analysis
- Autocorrelation diagnostics
- Volatility diagnostics
- Visualization

These tools allow us to formally evaluate the statistical behavior of financial return series.

## Load Data From CSV

In [14]:
df = pd.read_parquet('../data/sp500_cleaned.parquet').copy()
# preview Dataset
df.head()

Price,Open,High,Low,Close,Volume,log_returns
Date,,,,,,
2000-01-04,1455.219971,1455.219971,1397.430054,1399.420044,1009000000,-0.039099
2000-01-05,1399.420044,1413.270020,1377.680054,1402.109985,1085500000,0.001920
2000-01-06,1402.109985,1411.900024,1392.099976,1403.449951,1092300000,0.000955
2000-01-07,1403.449951,1441.469971,1400.729980,1441.469971,1225200000,0.026730
2000-01-10,1441.469971,1464.359985,1441.469971,1457.599976,1064800000,0.011128


## Load Cleaned Dataset

The dataset used in this notebook was cleaned and validated during the EDA phase.

This includes:

- Removal of anomalous observations
- Validation of trading dates
- Log return calculation
- Consistent datetime indexing

Using a pre-cleaned dataset ensures reproducibility and prevents duplication of preprocessing steps across notebooks.

In [15]:
# Dataset information
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 6627 entries, 2000-01-04 to 2026-05-12
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Open         6627 non-null   float64
 1   High         6627 non-null   float64
 2   Low          6627 non-null   float64
 3   Close        6627 non-null   float64
 4   Volume       6627 non-null   int64  
 5   log_returns  6627 non-null   float64
dtypes: float64(5), int64(1)
memory usage: 362.4 KB


In [16]:
# Check Index Type
type(df.index)

pandas.DatetimeIndex

In [17]:
# Summary Statistics
df.describe()

Price,Open,High,Low,Close,Volume,log_returns
count,6627.000000,6627.000000,6627.000000,6627.000000,6.627000e+03,6627.000000
mean,2331.242372,2344.485025,2317.010501,2331.638356,3.448504e+09,0.000245
std,1540.155489,1547.280450,1532.555194,1540.587187,1.525909e+09,0.012175
min,679.280029,695.270020,666.789978,676.530029,3.560700e+08,-0.127652
25%,1213.125000,1220.304993,1205.494995,1213.540039,2.344975e+09,-0.004729
50%,1552.359985,1557.250000,1545.900024,1552.479980,3.544720e+09,0.000641
75%,2933.469971,2947.185059,2919.550049,2938.109985,4.293290e+09,0.005881
max,7390.629883,7428.970215,7384.200195,7412.839844,1.145623e+10,0.109572


## Dataset Validation

Before performing statistical diagnostics, the dataset structure is briefly validated to confirm:

- Correct datetime indexing
- Presence of return series
- Numerical consistency
- Expected observation count

This step ensures the dataset is properly prepared for statistical testing.

# Distribution Diagnostics 

## 1) Skewness & Kurtosis

In [18]:
skewness = df['log_returns'].skew()
kurtosis = df['log_returns'].kurtosis()

print(f'Skewness: {skewness:.4f}')
print(f'Kurtosis: {kurtosis:.4f}')

Skewness: -0.3479
Kurtosis: 10.6444


## 📊 Interpretation of Skewness & Kurtosis Results

### Results

- **Skewness:** **-0.3479**  
- **Excess Kurtosis:** **10.6424**


### Interpretation of Skewness

The negative skewness indicates that the return distribution is slightly tilted toward larger downside moves.

This means:

- Negative returns tend to be more extreme than positive returns  
- Market declines are often sharper and more sudden than rallies  
- Downside tail risk is more pronounced  

This behavior is common in equity markets, where panic selling and systemic shocks can produce rapid drawdowns.


### Interpretation of Kurtosis

The excess kurtosis value of **10.6424** is substantially above zero.

The high kurtosis observed here indicates:

- The distribution contains **fat tails**
- Extreme returns occur far more frequently than predicted by Gaussian assumptions
- Market crashes and large rallies are statistically more common than standard models would expect


Financial return series often violate normal distribution assumptions due to:

- Fat tails  
- Extreme market events  
- Time-varying volatility  

These characteristics motivate the use of more advanced volatility and risk models in later stages of the project.

### Key Insight

> Extreme market movements occur more frequently than predicted by standard normal distribution assumptions.

# Q-Q Plot (Quantile-Quantile Plot)

In [19]:

# Generate Q-Q plot data
(osm, osr), (slope, intercept, r) = stats.probplot(
    df['log_returns'],
    dist='norm'
)

# Create Plotly figure
fig = go.Figure()

# Sample quantiles
fig.add_trace(
    go.Scatter(
        x=osm,
        y=osr,
        mode='markers',
        name='Observed Returns',
        marker=dict(size=4)
    )
)

# Reference normal line
fig.add_trace(
    go.Scatter(
        x=osm,
        y=slope * np.array(osm) + intercept,
        mode='lines',
        name='Normal Distribution Reference'
    )
)

# Layout
fig.update_layout(
    title='Q-Q Plot: S&P 500 Log Returns vs Normal Distribution',
    xaxis_title='Theoretical Quantiles',
    yaxis_title='Observed Quantiles',
    template='plotly_dark',
    height=700,
    width=900,
    legend_title=''
)

# Show plot
fig.show()

# Save figure
fig.write_image(
    "../reports/figures/sp500_qq_plot.png",
    width=1200,
    height=900,
    scale=2
)

# The Jarque-Bera Test

In [20]:
# Unpack the test results for clarity
jb_stat, p_value = stats.jarque_bera(df['log_returns'])

# Print results
print(f"Jarque-Bera Statistic: {jb_stat:,.2f}")
print(f"P-Value:               {p_value:.4f}")

# Add a simple logic check for the Null Hypothesis
alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject H0 - The distribution is NOT normal (Fat Tails confirmed).")
else:
    print("\nResult: Fail to reject H0 - The distribution appears normal.")

Jarque-Bera Statistic: 31,366.73
P-Value:               0.0000

Result: Reject H0 - The distribution is NOT normal (Fat Tails confirmed).


### 📊 Jarque-Bera Normality Test

The Jarque-Bera test formally assesses whether the data has skewness and kurtosis matching a normal distribution.

**Results:**
- **Jarque-Bera Statistic:** 31,350.69(Very large)
- **P-Value:** < 0.05

**Conclusion:** We strongly reject the null hypothesis of normality.

**Key Insight**
> Because S&P 500 returns exhibit **fat tails** and **volatility clustering**:
- Traditional VaR/ES models assuming normality **significantly underestimate tail risk**.
- This can lead to insufficient capital buffers during market stress.
- **GARCH-family models** address this by explicitly modeling time-varying volatility and clustering — providing more accurate risk forecasts.

These diagnostics directly justify moving from simple statistical models to more sophisticated volatility modeling in Notebook 05.

# Serial Dependence Diagnostics 

## ACF of Returns

In [21]:
# Compute ACF + confidence intervals
acf_values, confint = acf(
    df['log_returns'],
    nlags=40,
    alpha=0.05
)

lags = np.arange(len(acf_values))

# Confidence interval around zero
ci_upper = confint[:,1] - acf_values
ci_lower = acf_values - confint[:,0]

fig = go.Figure()

# Bars
fig.add_trace(
    go.Bar(
        x=lags,
        y=acf_values,
        name='ACF'
    )
)

# Upper confidence band
fig.add_trace(
    go.Scatter(
        x=lags,
        y=ci_upper,
        mode='lines',
        line=dict(dash='dash'),
        name='95% CI'
    )
)

# Lower confidence band
fig.add_trace(
    go.Scatter(
        x=lags,
        y=-ci_lower,
        mode='lines',
        line=dict(dash='dash'),
        showlegend=False
    )
)

# Zero line
fig.add_hline(y=0, line_dash='dash')

fig.update_layout(
    title='ACF of S&P 500 Log Returns',
    xaxis_title='Lag',
    yaxis_title='Autocorrelation',
    template='plotly_dark'
)

fig.show()

### 📊 Autocorrelation Analysis (ACF) — Log Returns

The Autocorrelation Function (ACF) examines whether current log returns are correlated with their past values at different lags.

**Observations**
- Most lags show autocorrelation values very close to zero.
- A small but statistically significant **negative autocorrelation** appears at **lag 1**.
- No slow decay or strong persistent patterns are visible across higher lags.

**Interpretation**

The S&P 500 daily log returns exhibit **very weak serial correlation**. 

While the slight negative lag-1 autocorrelation is statistically significant, its economic magnitude is small. This aligns with the **weak-form Efficient Market Hypothesis** — past price movements provide limited predictive power for future directional moves.

**Important Nuance**

> Absence of strong autocorrelation in **returns** does **not** mean the market is purely random.

Financial time series frequently show much stronger dependence in:
- **Squared returns** (volatility clustering)
- **Absolute returns**
- Conditional variance

This is why we often observe near-zero ACF in raw returns but significant autocorrelation in squared returns — a key stylized fact that motivates **GARCH-type models**.

**Key Insight**

> Daily returns are largely unpredictable from past returns alone, but **volatility** (risk) itself tends to persist over time.

# ACF / PACF of Squared Returns

In [22]:
# Create squared returns 
df['squared_returns'] = df['log_returns'] ** 2

# Check results 
df.head()

Price,Open,High,Low,Close,Volume,log_returns,squared_returns
Date,,,,,,,
2000-01-04,1455.219971,1455.219971,1397.430054,1399.420044,1009000000,-0.039099,1.528746e-03
2000-01-05,1399.420044,1413.270020,1377.680054,1402.109985,1085500000,0.001920,3.687698e-06
2000-01-06,1402.109985,1411.900024,1392.099976,1403.449951,1092300000,0.000955,9.124486e-07
2000-01-07,1403.449951,1441.469971,1400.729980,1441.469971,1225200000,0.026730,7.144902e-04
2000-01-10,1441.469971,1464.359985,1441.469971,1457.599976,1064800000,0.011128,1.238285e-04


### 📊 Squared Returns

To study volatility behavior, log returns are squared.

Unlike raw returns, squared returns remove market direction and focus on the **magnitude of price movements**.

This allows us to examine whether periods of:

- Large market movements  
- Elevated risk  
- Relative market calm  

tend to persist over time.

If squared returns exhibit autocorrelation, this suggests **volatility clustering** — a defining characteristic of financial markets.

**Why this matters**

Traditional return autocorrelation focuses on **directional predictability**.

Squared returns instead focus on **risk persistence**.

A market may show little predictability in price direction while still displaying persistent and forecastable volatility.

In [29]:
# Compute acf for returns 
lags = 40   # 2 Months 
acf_vals = acf(df['squared_returns'].dropna(), nlags= lags)

# Confidence Interval 
conf = 1.96 / np.sqrt(len(df)) # 95% Confidence and 1.96 * std 

# Plot
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=list(range(len(acf_vals))),
        y=acf_vals,
        name='ACF'
    )
)

# Confidence bands
fig.add_hline(y=conf, line_dash='dash')
fig.add_hline(y=-conf, line_dash='dash')
fig.add_hline(y=0)

fig.update_layout(
    title='ACF of Squared S&P 500 Log Returns',
    xaxis_title='Lag',
    yaxis_title='Autocorrelation',
    showlegend=False
)

fig.show()
fig.write_image(
    "../reports/figures/acf_squared_returns.png",
    width=1400,
    height=700,
    scale=2
)

### 📊 ACF of Squared Returns — Volatility Persistence

The Autocorrelation Function (ACF) of squared returns examines whether **volatility is correlated with its own past values**.

Unlike the ACF of raw returns, which measures directional dependence, squared-return autocorrelation focuses on the **persistence of market risk**.

**Lag Selection**

The number of lags determines how far back volatility dependence is examined.

For daily financial data, using **20–40 lags is common practice**:

- **20 lags** ≈ roughly **1 trading month**  
- **40 lags** ≈ roughly **2 trading months**  

This range provides a practical balance:

- Long enough to detect **volatility persistence and clustering**  
- Short enough to avoid excessive statistical noise and difficult interpretation  

In this analysis, **40 lags** were used to evaluate whether volatility dependence persists beyond short-term market fluctuations.

**Observations**

- A large number of lags exceed the confidence intervals  
- Autocorrelation remains statistically significant across many lags  
- The autocorrelation gradually decays over time rather than disappearing immediately  

**Interpretation**

This pattern provides strong evidence of:

- **Volatility clustering**  
- **Time-varying variance (heteroskedasticity)**  
- Persistent dependence in return magnitude  

The gradual decay is particularly important.

If volatility were independent over time, autocorrelation would fluctuate randomly around zero.

Instead, the slow decline suggests that volatility exhibits **memory**, where periods of elevated market risk tend to persist before eventually reverting toward calmer conditions.

This behavior is one of the most widely documented **stylized facts of financial markets**.

**Key Insight**

> While market direction may appear difficult to predict, **volatility itself displays persistent and measurable structure**.

# PACF of Squared Returns

In [31]:
# Compute PACF
pacf_vals = pacf(
    df['squared_returns'].dropna(),
    nlags = 40,
    method = 'ywm'
) 

# Plot
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=list(range(len(pacf_vals))),
        y=pacf_vals,
        name='PACF'
    )
)

# Confidence bands
fig.add_hline(y=conf, line_dash='dash')
fig.add_hline(y=-conf, line_dash='dash')
fig.add_hline(y=0)

fig.update_layout(
    title='PACF of Squared S&P 500 Log Returns',
    xaxis_title='Lag',
    yaxis_title='Partial Autocorrelation',
    showlegend=False
)

fig.show()
fig.write_image(
    "../reports/figures/pacf_squared_returns.png",
    width=1400,
    height=700,
    scale=2
)

### 📊 PACF of Squared Returns — Direct Volatility Dependence

The Partial Autocorrelation Function (PACF) measures the **direct relationship** between squared returns and prior lags after controlling for intermediate lag effects.

While ACF captures total dependence across lags, PACF isolates the **direct contribution** of individual lags.

**Observations**

- Statistically significant autocorrelation appears primarily in the **first 4–5 lags**  
- After these early lags, partial autocorrelation declines rapidly and moves within the confidence intervals  
- No long sequence of significant direct lag effects is observed  

**Interpretation**

This result suggests that volatility dependence is driven mainly by **recent market activity**.

The first few lagged volatility observations contain most of the explanatory information, while more distant lags contribute relatively little once earlier volatility effects are accounted for.

This explains an important difference between the ACF and PACF results:

- **ACF** shows persistent volatility dependence across many lags  
- **PACF** indicates that the strongest **direct** effects occur primarily at short horizons  

This combination is commonly observed in financial time series and is consistent with **ARCH/GARCH-type volatility processes**.

**Why this matters**

The PACF results suggest that volatility is not influenced equally by all historical periods.

Instead, recent volatility carries the strongest direct influence on future volatility behavior.

This provides additional statistical justification for using **volatility models that explicitly account for short-term persistence and changing variance**.

**Key Insight**

> Financial volatility often displays **short-term direct dependence combined with longer-term persistence**, a characteristic that motivates ARCH and GARCH volatility modeling.